# 基礎編10
このノートブックでは、サブコントラクト呼び出しに関する例を示します。

## 概要

スマートコントラクトの中から、別のスマートコントラクトを呼び出すことができます。  
呼び出される方のコントラクトをサブコントラクトと呼びます。  
呼び出しは同期的に行われ、返り値を得ることができます。

## 準備

In [1]:
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## スマートコントラクトのデプロイ

サブコントラクトをデプロイします。

In [2]:
var subcid = await deploySmartContract({ a: 'number' }, function basic10sub(a){
    return a + ': ' + getTxno() + ': calling TX is ' + getCallingTxno();
});
console.log('subcontract ID:', subcid);

subcontract ID: c060278367


サブコントラクトを呼び出すコントラクトをデプロイします。  
(__subcid__はサブコントラクトのIDに置換されます)

In [3]:
var cid = await deploySmartContract(function basic10(){
    var cnt = openContract('__subcid__'); // サブコントラクトを開きます
    var res1 = cnt.call({a:1}); // サブコントラクトを引数を指定して呼び出します
    var res2 = cnt.call({a:2}); // 呼び出しは繰り返すことができます
    return [res1,res2];
}, {
    replacers: [['__subcid__', subcid ]]
});

サブコントラクトを実行できるように権限の設定をします。

In [4]:
await rpc.call(adminWallet, 'c1update', { id: subcid, prop: 'accessible_to', value: [ cid ] });

{
  txno: 701783,
  txid: 'xpW2zGpVphm7MjRAbkvGJN2guP2GbZ2cudzMuRnWe2vq2',
  status: 'ok',
  value: null
}

念のため設定を確認します。

In [5]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_contract', id: cid });
console.log(resp.value);
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_contract', id: subcid });
console.log(resp.value);

{
  id: [ 'c076131498', 'basic10@c.TDSL.H011' ],
  frozen: null,
  name: 'basic10',
  domain: [ 'd93319890', '@c.TDSL.H011' ],
  description: '',
  argtypes: {},
  code: "    var cnt = openContract('c060278367');     var res1 = cnt.call({a:1});     var res2 = cnt.call({a:2});     return [res1,res2];",
  mask: {},
  maxsteps: 0,
  a_txno: 701780,
  c_txno: 701780,
  num_transactions: 0,
  last_active: 1753420798349,
  created_at: 1753420798349,
  accessible_to: [ [ 'u58281888', 'jupyter@c.TDSL.H011' ] ],
  callable_to: [ 'all' ],
  disclosed_to: [],
  groups: [],
  managed: true,
  accessible: true
}
{
  id: [ 'c060278367', 'basic10sub@c.TDSL.H011' ],
  frozen: null,
  name: 'basic10sub',
  domain: [ 'd93319890', '@c.TDSL.H011' ],
  description: '',
  argtypes: { a: 'number' },
  code: "    return a + ': ' + getTxno() + ': calling TX is ' + getCallingTxno();",
  mask: {},
  maxsteps: 0,
  a_txno: 701777,
  c_txno: 701777,
  num_transactions: 0,
  last_active: 1753420796897,
  created_at

## 実行

In [6]:
var resp = await rpc.call(adminWallet, cid);
console.log(resp);

{
  txno: 701784,
  txid: 'xJ68AWn86Z78dpekeoDQmDevmEDAXLmYLKWGvKKpFCo48',
  status: 'ok',
  value: [
    '1: 701785: calling TX is 701784',
    '2: 701786: calling TX is 701784'
  ]
}


サブコントラクトのそれぞれの実行には異なるトランザクション番号が振られています。  
もういちど実行します。

In [7]:
var resp = await rpc.call(adminWallet, cid);
console.log(resp);

{
  txno: 701787,
  txid: 'xnXvRyPJgwwraxYBf8beG7GadHdCA6gGazcCNbFcGpPNt',
  status: 'ok',
  value: [
    '1: 701788: calling TX is 701787',
    '2: 701789: calling TX is 701787'
  ]
}
